# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuchau0802/run-starter-notebooks/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

I’m treating Lane 2 as a ranking task because a content editor needs to know which pages to review first. The output is a queue ordered by estimated risk of losing search visibility, with a short reason for each recommendation. The editor reviews the top 20 pages, checks whether the decline is sustained, and decides whether to refresh or keep monitoring each page.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

In the starter notebook, I’ll create a binary column called decline_proxy. It is 1 when the page’s observed trend_direction is down and 0 otherwise. For the capstone, my proposed target is an observed drop of more than 20% in search impressions in the 30 days after the review date compared with the 30 days before it. I’ll require sufficient impression volume and complete data for both windows. Only information available by the review date can go into the score. The starter proxy shows how the target column is structured. A refresh decision still requires editorial review.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

My primary metric is Precision@20, the share of the first 20 pages in the queue that show the measured decline. I chose 20 because that is the number of pages the editor plans to review. Success means finding more declining pages in those 20 slots than the stale-and-visible rule does on the same held-out data, with the outcome measured after the review date. The starter snapshot score below is a descriptive check because its 90-day impression count overlaps the period used to mark decline.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

One row represents one content page in the starter snapshot. For the capstone, one row would represent one page at one review date. The preview shows measured page fields and the decline proxy. I use the page ID to verify uniqueness and leave it out of the preview.

In [4]:
import pandas as pd

url = (
    "https://raw.githubusercontent.com/vuchau0802/run-starter-notebooks/"
    "main/data/raw/content_refresh_anonymized.csv"
)
columns = [
    "content_id", "days_since_last_update", "impressions_90d",
    "impressions_prev_30d", "trend_direction",
]
pages = pd.read_csv(url, usecols=columns)

assert pages["content_id"].notna().all() and pages["content_id"].is_unique
pages["decline_proxy"] = pages["trend_direction"].eq("down").astype(int)

print(f"{len(pages):,} rows, one per content page")
display(pages[["days_since_last_update", "impressions_prev_30d", "decline_proxy"]].head())

# Put pages that meet the rule first, then rank by impressions within each group.
pages["stale_visible"] = (
    pages["days_since_last_update"].ge(180)
    & pages["impressions_90d"].ge(500)
)
top20 = pages.sort_values(
    ["stale_visible", "impressions_90d"], ascending=[False, False]
).head(20)

print(f"Starter snapshot baseline Precision@20: {top20['decline_proxy'].mean():.3f}")

30,000 rows, one per content page


,days_since_last_update,impressions_prev_30d,decline_proxy
0,20,987,1
1,25,5915,1
2,20,6089,1
3,22,4206,0
4,14,6452,1


Starter snapshot baseline Precision@20: 0.900


## 5. Why ML beats a fixed rule here

A fixed rule uses the same thresholds for every page. A page just below 500 impressions can be overlooked even if its search visibility has been falling steadily. A model could learn how update age, search visibility, and past changes in impressions work together. In ML-01, my hand rule beat both trees on the held-out clients, so the rule is a serious baseline. I’ll keep it if the model fails to improve Precision@20 on held-out data.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.